# CA Affordable Housing Tax Credit Intelligence Platform
## Notebook 02 — Data Enrichment & Final Merge

**Input:** `data/processed/ctcac_clean.csv` (5,496 CA LIHTC projects, cleaned — after re-running `01_data.ipynb`)

**Enrichments:**
1. FRED PPI — annual construction cost index + YoY trend features
2. HUD Fair Market Rents — 2BR FMR by county × year
3. Census ACS 5-Year (2022) — county-level income, rent, poverty

**Output:** `data/processed/lihtc_ca_clean.parquet` — fully merged, model-ready dataset

In [ ]:
import warnings
import requests
import numpy as np 
import pandas as pd
import os 

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

## ============Your API Key =============================
CENSUS_API_KEY = 'Your API Key Here' # Replace with your actual API key

#CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY')
if not CENSUS_API_KEY:
    raise ValueError('Set CENSUS_API_KEY env var: export CENSUS_API_KEY=your_key')

print('Setup complete')

Setup complete


#### 1. Load Cleaned CTCAC Data

In [14]:
df = pd.read_csv('../data/processed/ctcac_clean.csv')
# Convert from float to nullable int for join keys
df['pis_year'] = df['pis_year'].astype('Int64')

print(f'Leaded: {df.shape[0]} rows X {df.shape[1]} columns')
print(f'Placed in Service Year: {df["pis_year"].min()} - {df["pis_year"].max()}')
df.head()

Leaded: 5496 rows X 27 columns
Placed in Service Year: 1990 - 2030


,app_number,project_name,county,city,census_tract,developer,pis_year,credit_type,construction_type,housing_type,...,three_plus_br_pct,deep_ami_pct,low_ami_pct,mid_ami_pct,high_ami_pct,credit_per_unit,implied_eligible_basis,implied_basis_per_unit,total_state_award,annual_federal_award
0,CA-1989-279,Tres Palmas Village 90-001,Imperial,Brawley,NaN,Opportunity Housing,1990,9%,New Construction,Large Family,...,0.00,0.00,0.00,0.00,0.00,"1,784.38","1,090,455.56","19,826.46",NaN,"98,141.00"
1,CA-1990-011,Villa Los Robles,Los Angeles,Pasadena,4619.01,"CKMP, Inc",1992,9%,New Construction,Large Family,...,0.00,0.00,0.00,0.00,0.00,"9,168.62","814,988.89","101,873.61","357,576.00","73,349.00"
2,CA-1990-012,Casa Loma Apartments,Los Angeles,Los Angeles,2091.04,New Economics for Women,1993,9%,New Construction,Large Family,...,0.29,0.00,0.00,0.00,0.00,"13,786.32","16,849,944.44","153,181.31",NaN,"1,516,495.00"
3,CA-1990-014,San Pedro Gardens,Santa Clara,Morgan Hill,5123.10,"S.P.G. Housing, Inc.",1992,9%,New Construction,Large Family,...,0.00,0.00,0.00,0.00,0.00,"13,877.00","3,627,973.86","181,398.69",NaN,"277,540.00"
4,CA-1990-018,Yucaipa Terrace,San Bernardino,Yucaipa,87.04,"Housing Partners, Inc.",1991,9%,New Construction,Senior,...,0.00,0.00,0.00,0.00,0.00,"5,673.86","3,215,188.89","63,042.92",NaN,"289,367.00"


#### 2. FRED PPI - Construction Cost Index Features
Series: WPUSI012011 (Producer Price Index, Construction Materials)

We compute annual averages then derive three temporal features:
- `ppi_at_allocation` — average PPI in the placed-in-service year
- `ppi_yoy_change` — year-over-year % change in annual PPI
- `ppi_2yr_trend` — 2-year rolling direction (rising cost environment = 1, flat/falling = 0)

In [15]:
ppi_raw = pd.read_csv('fred_ppi.csv', parse_dates=['observation_date'])
#add column names
ppi_raw.columns = ['date', 'ppi']
#ppi_raw

# Annual average/mean PPI per year
ppi_annual = (ppi_raw
              .assign(year=lambda x: x['date'].dt.year) #strip month/day, keep year
              .groupby('year')['ppi']
              .mean() #calcualte annual average PPI
              .rename('ppi_at_allocation')
              .reset_index())

# Add column to calcualte year over year PPI % change
ppi_annual['ppi_yoy_change'] = ppi_annual['ppi_at_allocation'].pct_change() * 100

# 2-year rolling trend: is PPI now higher than it was 2 years ago?
# If current PPI is higher than 2 years ago, trend is positive (1), else negative (0)
ppi_annual['ppi_2yr_trend'] = (ppi_annual['ppi_at_allocation'] 
                               > ppi_annual['ppi_at_allocation'].shift(2)).astype(float)

print(f'PPI Annual: {ppi_annual['year'].min()} - {ppi_annual['year'].max()}')
ppi_annual.tail(10)

PPI Annual: 1990 - 2026


,year,ppi_at_allocation,ppi_yoy_change,ppi_2yr_trend
27,2017,221.64,3.62,1.00
28,2018,235.75,6.37,1.00
29,2019,235.72,-0.01,1.00
30,2020,239.18,1.47,1.00
31,2021,303.41,26.85,1.00
32,2022,341.53,12.56,1.00
33,2023,331.78,-2.85,1.00
34,2024,328.52,-0.98,0.00
35,2025,337.92,2.86,1.00
36,2026,349.95,3.56,1.00


In [16]:
# Join PPI data to CTCAC data by pis_year
#rename year to pis_year in ppi_annual for join, then left join to preserve all CTCAC records
df = df.merge(ppi_annual.rename(columns={'year': 'pis_year'}), 
              on='pis_year', how='left')

# Check how many ppi_at_allocation values are missing after the join, and calculate match rate
# See if the PPI values cover the full range of placed in service years in the CTCAC data
match_rate = df['ppi_at_allocation'].notna().mean() * 100
print(f'PPI join match rate: {match_rate:.1f}%')
print(f'Missing PPI rows: {df["ppi_at_allocation"].isna().sum()}')

PPI join match rate: 93.7%
Missing PPI rows: 345


---
### 3. HUD Fair Market Rents — 2BR FMR by County × Year

The FMR wide table has one row per county and columns `fmrYY_2` for each year.
We melt to long format, then join by county + pis_year.

FMR years available: 1983–2026 (columns `fmr83_2` through `fmr26_2`).
The CTCAC data runs 2000–2025, so coverage is complete.

In [17]:
fmr_raw = pd.read_csv('FMR_ALL_1983_2026.csv', encoding = 'latin-1')
print(f'FMR Raw: {fmr_raw.shape[0]} rows X {fmr_raw.shape[1]} columns')

fmr_raw.head()

FMR Raw: 4764 rows X 310 columns


,fips,fips2024,census_region,state,county,cousub,areaname26,name,msa26,fmr26_0,...,fmr85_3,fmr85_4,fmr85,msa83,fmr83_0,fmr83_1,fmr83_2,fmr83_3,fmr83_4,fmr83
0,100199999,100199999,3.00,1,1,99999,"Montgomery, AL MSA",Autauga County,METRO33860M33860,$860,...,344.00,382.00,45.00,"5,240.00",186.00,227.00,269.00,332.00,370.00,45.00
1,100399999,100399999,3.00,1,3,99999,"Daphne-Fairhope-Foley, AL MSA",Baldwin County,METRO19300M19300,"$1,094",...,393.00,439.00,45.00,"5,160.00",217.00,257.00,309.00,380.00,425.00,45.00
2,100599999,100599999,3.00,1,5,99999,"Barbour County, AL",Barbour County,NCNTY01005N01005,$576,...,387.00,426.00,45.00,"10,000.00",212.00,257.00,300.00,374.00,413.00,45.00
3,100799999,100799999,3.00,1,7,99999,"Birmingham-Hoover, AL HUD Metro FMR Area",Bibb County,METRO13820M13820,"$1,024",...,400.00,447.00,45.00,"10,000.00",218.00,265.00,312.00,387.00,433.00,45.00
4,100999999,100999999,3.00,1,9,99999,"Birmingham-Hoover, AL HUD Metro FMR Area",Blount County,METRO13820M13820,"$1,024",...,417.00,462.00,45.00,"1,000.00",229.00,280.00,327.00,404.00,448.00,45.00


In [18]:
fmr_raw.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4764 entries, 0 to 4763
Data columns (total 310 columns):
 #    Column         Non-Null Count  Dtype  
---   ------         --------------  -----  
 0    fips           4764 non-null   int64  
 1    fips2024       4764 non-null   int64  
 2    census_region  4754 non-null   float64
 3    state          4764 non-null   int64  
 4    county         4764 non-null   int64  
 5    cousub         4764 non-null   int64  
 6    areaname26     4764 non-null   str    
 7    name           4763 non-null   str    
 8    msa26          4764 non-null   str    
 9    fmr26_0        4764 non-null   str    
 10   fmr26_1        4764 non-null   str    
 11   fmr26_2        4764 non-null   str    
 12   fmr26_3        4764 non-null   str    
 13   fmr26_4        4764 non-null   str    
 14   msa25          4763 non-null   str    
 15   fmr25_0        4763 non-null   float64
 16   fmr25_1        4763 non-null   float64
 17   fmr25_2        4763 non-null   float64
 18

In [19]:
# Filter to California (state FIPS=6)
ca_fmr = fmr_raw[fmr_raw['state'] == 6].copy()

# Strip " County" suffix from name to match CTCAC county names
ca_fmr['county_clean'] = ca_fmr['name'].str.replace(' County', '', regex=False).str.strip()

print(f'CA FMR counties: {len(ca_fmr)}')
print(ca_fmr['county_clean'].tolist())
#ca_fmr

CA FMR counties: 58
['Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa', 'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn', 'Humboldt', 'Imperial', 'Inyo', 'Kern', 'Kings', 'Lake', 'Lassen', 'Los Angeles', 'Madera', 'Marin', 'Mariposa', 'Mendocino', 'Merced', 'Modoc', 'Mono', 'Monterey', 'Napa', 'Nevada', 'Orange', 'Placer', 'Plumas', 'Riverside', 'Sacramento', 'San Benito', 'San Bernardino', 'San Diego', 'San Francisco', 'San Joaquin', 'San Luis Obispo', 'San Mateo', 'Santa Barbara', 'Santa Clara', 'Santa Cruz', 'Shasta', 'Sierra', 'Siskiyou', 'Solano', 'Sonoma', 'Stanislaus', 'Sutter', 'Tehama', 'Trinity', 'Tulare', 'Tuolumne', 'Ventura', 'Yolo', 'Yuba']


We use 2-bedroom FMR as a proxy for local rental market conditions, as it is the most commonly used benchmark in affordable housing analysis. While project unit mixes vary, 2BR rents provide a stable and comparable reference across regions.

In [20]:
# Select county identifier + all 2BR FMR columns
# fmrxx_x where xx is the year and x is the number of bedrooms 
# (e.g., fmr23_2br for 2023 2-bedroom FMR)
fmr_cols_2br = [c for c in ca_fmr.columns if 
                c.startswith('fmr') and c.endswith('_2')]
print(fmr_cols_2br)
fmr_wide = ca_fmr[['county_clean'] + fmr_cols_2br]
#fmr_wide.head()
fmr_long = fmr_wide.melt(id_vars='county_clean',
                         value_vars = fmr_cols_2br,
                         var_name='fmr_col',
                         value_name='fmr_2br')
fmr_long.head()

['fmr26_2', 'fmr25_2', 'fmr24_2', 'fmr23_2', 'fmr22_2', 'fmr21_2', 'fmr20_2', 'fmr19_2', 'fmr18_2', 'fmr17_2', 'fmr16_2', 'fmr15_2', 'fmr14_2', 'fmr13_2', 'fmr12_2', 'fmr11_2', 'fmr10_2', 'fmr09_2', 'fmr08_2', 'fmr07_2', 'fmr06_2', 'fmr05_2', 'fmr04_2', 'fmr03_2', 'fmr02_2', 'fmr01_2', 'fmr00_2', 'fmr99_2', 'fmr98_2', 'fmr97_2', 'fmr96_2', 'fmr95_2', 'fmr94_2', 'fmr93_2', 'fmr92_2', 'fmr91_2', 'fmr90_2', 'fmr89_2', 'fmr88_2', 'fmr87_2', 'fmr86_2', 'fmr85_2', 'fmr83_2']


,county_clean,fmr_col,fmr_2br
0,Alameda,fmr26_2,"$2,912"
1,Alpine,fmr26_2,"$1,529"
2,Amador,fmr26_2,"$1,698"
3,Butte,fmr26_2,"$1,625"
4,Calaveras,fmr26_2,"$1,542"


In [21]:
# Parse year from column name: fmr26_2 → 2026, fmr00_2 → 2000, fmr99_2 → 1999
def parse_fmr_year(col):
    # Extract the two-digit year from the column name
    # Get the last two digits of the year,ex.from fmr26_2 get 26
    # from fmr00_2 get 00, from fmr99_2 get 99
    # FMR data from 1983-2026, we can use a cutoff of 50 for future years
    yy = int(col.replace('fmr', '').replace('_2','')) 
    return 2000+yy if yy <=50 else 1900+yy
fmr_long['year'] = fmr_long['fmr_col'].apply(parse_fmr_year)
#fmr_long.head()
# Select only relevant columns for join: county_clean, fmr_year, fmr_2br
fmr_long = fmr_long[['county_clean', 'year', 'fmr_2br']]

# Normalize fmr_2br: older year columns are floats, recent years use "$2,912" strings
fmr_long['fmr_2br'] = (fmr_long['fmr_2br']
                       .astype(str)
                       .str.replace('$','', regex=False)
                       .str.replace(',','', regex=False)
                       .str.strip())
# Convert to numeric, coercing errors to NaN 
# (e.g., if there are any non-convertible strings)
fmr_long['fmr_2br'] = pd.to_numeric(fmr_long['fmr_2br'], errors='coerce')

print(f'FMR long shape: {fmr_long.shape}')
print(f'Years: {fmr_long["year"].min()} – {fmr_long["year"].max()}')
fmr_long.head()                        

FMR long shape: (2494, 3)
Years: 1983 – 2026


,county_clean,year,fmr_2br
0,Alameda,2026,"2,912.00"
1,Alpine,2026,"1,529.00"
2,Amador,2026,"1,698.00"
3,Butte,2026,"1,625.00"
4,Calaveras,2026,"1,542.00"


In [22]:
# Normalize CTCAC county to Title Case for join
df['county_join'] = df['county'].str.strip().str.title()

# Join FMR by county + pis_year
df = df.merge(
    fmr_long.rename(columns={'county_clean': 'county_join', 'year': 'pis_year'}),
    on=['county_join', 'pis_year'],
    how='left')

match_rate = df['fmr_2br'].notna().mean() * 100
print(f'FMR join match rate: {match_rate:.2f}%')
print(f'Missing FMR rows: {df["fmr_2br"].isna().sum()}')

# Drop join helper column
df = df.drop(columns=['county_join'])

FMR join match rate: 93.72%
Missing FMR rows: 345


In [23]:
# Diagnose any unmatched counties
unmatched = df[df['fmr_2br'].isna()]['county'].value_counts()
if len(unmatched)>0:
    print('Counties with missing FMR (check spelling):')
    print(unmatched)
else:
    print('All counties matched successfully!')

Counties with missing FMR (check spelling):
county
Los Angeles        89
San Diego          39
Santa Clara        25
San Francisco      22
Alameda            21
Riverside          17
Santa Cruz         12
Contra Costa       11
San Mateo          11
Sacramento          9
Orange              7
Fresno              7
Sonoma              6
Placer              6
Ventura             6
Santa Barbara       6
Butte               5
San Joaquin         4
Humboldt            4
San Luis Obispo     3
Tulare              3
Solano              3
El Dorado           3
San Bernardino      3
Marin               3
Lake                2
Kings               2
San Benito          2
Kern                2
Merced              1
Del Norte           1
Madera              1
Stanislaus          1
Yolo                1
Monterey            1
Mendocino           1
Tehama              1
Sutter              1
Nevada              1
Mono                1
Napa                1
Name: count, dtype: int64


---
### 4. Census ACS 5-Year Estimates — County-Level Economic Context

We pull 2022 ACS 5-year estimates for all California counties via the Census API.
These are used as static county-level enrichment features (not year-varying).

Data link: https://api.census.gov/data/2022/acs/acs5

Variable link: https://api.census.gov/data/2022/acs/acs5/variables.json

Example, State code and Reference: https://api.census.gov/data/2022/acs/acs5.html

Census API Guide: https://www.census.gov/data/what-is-data-census-gov/guidance-for-data-users/how-to-materials-for-using-the-census-api.html

| Variable | ACS Code | What it captures |
|---|---|---|
| `county_median_income` | B19013_001E | Median household income → AMI limits |
| `county_median_rent` | B25064_001E | Median gross rent → market rent pressure |
| `county_poverty_count` | B17001_002E | Below poverty count → affordability demand |
| `county_rental_vacancy` | B25004_002E | Rental vacancy count → housing supply tightness |

In [24]:
# API call to get ACS data for CA counties
ACS_VARIABLES = [
    'NAME',
    'B19013_001E',   # median household income
    'B25064_001E',   # median gross rent
    'B17001_002E',   # population below poverty
    'B25004_002E']  # rental vacancy count

url = 'https://api.census.gov/data/2022/acs/acs5'
parameters = {
    'get':','.join(ACS_VARIABLES),
    'for':'county:*',
    'in':'state:06',
    'key': CENSUS_API_KEY}

response = requests.get(url, params=parameters)
response.raise_for_status() # Raise an error for bad status codes
data = response.json()
acs_df = pd.DataFrame(data[1:], columns=data[0])

print(f'ACS data shape: {acs_df.shape}')
print(f'ACS rows: {acs_df.shape[0]} (should be 58 CA counties)')
acs_df.head()

ACS data shape: (58, 7)
ACS rows: 58 (should be 58 CA counties)


,NAME,B19013_001E,B25064_001E,B17001_002E,B25004_002E,state,county
0,"Alameda County, California",122488,2229,149843,13784,06,001
1,"Alpine County, California",101125,-666666666,208,19,06,003
2,"Amador County, California",74853,1257,2945,107,06,005
3,"Butte County, California",66085,1280,38015,2728,06,007
4,"Calaveras County, California",77526,1621,5915,179,06,009


In [25]:
# Clean ACS data
acs_df = acs_df.rename(columns={
    'B19013_001E': 'county_median_income',
    'B25064_001E': 'county_median_rent',
    'B17001_002E': 'county_poverty_count',
    'B25004_002E': 'county_rental_vacancy'})

#Extract clean county name for NAME column (e.g., "Alameda County, California" → "Alameda")
acs_df['county_join'] = (acs_df['NAME']
                           .str.split(',').str[0] #split on comma and take first part -> Alameda County
                           .str.replace(' County', '', regex=False) #remove " County" suffix -> Alameda
                           .str.strip()
                           .str.title())
#acs_df.head()

#Convert numeric columns (Census returns strings; -666666666 = missing)
num_cols = ['county_median_income', 'county_median_rent', 'county_poverty_count', 'county_rental_vacancy']
for col in num_cols:
    acs_df[col] = pd.to_numeric(acs_df[col], errors='coerce')
    acs_df.loc[acs_df[col] < 0, col] = np.nan #replace Census sentinel values

acs_df = acs_df[['county_join'] + num_cols]

print(f'ACS counties: {len(acs_df)}')
print(acs_df.describe())
acs_df.head()

ACS counties: 58
       county_median_income  county_median_rent  county_poverty_count  \
count                 58.00               57.00                 58.00   
mean              82,966.60            1,536.46             80,780.55   
std               24,985.52              488.75            187,968.71   
min               47,317.00              796.00                208.00   
25%               64,143.25            1,146.00              5,894.00   
50%               76,147.50            1,289.00             26,920.00   
75%               98,693.75            1,874.00             74,922.50   
max              153,792.00            2,805.00          1,343,978.00   

       county_rental_vacancy  
count                  58.00  
mean                4,314.66  
std                10,751.73  
min                     0.00  
25%                   247.50  
50%                 1,060.00  
75%                 3,342.25  
max                75,430.00  


,county_join,county_median_income,county_median_rent,county_poverty_count,county_rental_vacancy
0,Alameda,"122,488.00","2,229.00","149,843.00","13,784.00"
1,Alpine,"101,125.00",NaN,208.00,19.00
2,Amador,"74,853.00","1,257.00","2,945.00",107.00
3,Butte,"66,085.00","1,280.00","38,015.00","2,728.00"
4,Calaveras,"77,526.00","1,621.00","5,915.00",179.00


In [26]:
# Join ACS onto CTCAC by county
df['county_join'] = df['county'].str.strip().str.title()

df = df.merge(acs_df, on='county_join', how='left')

match_rate = df['county_median_income'].notna().mean() * 100
print(f'ACS join match rate: {match_rate:.2f}%')
print(f'Missing income rows: {df["county_median_income"].isna().sum()}')

df = df.drop(columns=['county_join'])
df.head()

ACS join match rate: 100.00%
Missing income rows: 0


,app_number,project_name,county,city,census_tract,developer,pis_year,credit_type,construction_type,housing_type,...,total_state_award,annual_federal_award,ppi_at_allocation,ppi_yoy_change,ppi_2yr_trend,fmr_2br,county_median_income,county_median_rent,county_poverty_count,county_rental_vacancy
0,CA-1989-279,Tres Palmas Village 90-001,Imperial,Brawley,NaN,Opportunity Housing,1990,9%,New Construction,Large Family,...,NaN,"98,141.00",119.62,NaN,0.00,529.00,"53,847.00",961.00,"36,092.00",684.00
1,CA-1990-011,Villa Los Robles,Los Angeles,Pasadena,4619.01,"CKMP, Inc",1992,9%,New Construction,Large Family,...,"357,576.00","73,349.00",122.51,1.77,1.00,804.00,"83,411.00","1,805.00","1,343,978.00","75,430.00"
2,CA-1990-012,Casa Loma Apartments,Los Angeles,Los Angeles,2091.04,New Economics for Women,1993,9%,New Construction,Large Family,...,NaN,"1,516,495.00",128.60,4.97,1.00,829.00,"83,411.00","1,805.00","1,343,978.00","75,430.00"
3,CA-1990-014,San Pedro Gardens,Santa Clara,Morgan Hill,5123.10,"S.P.G. Housing, Inc.",1992,9%,New Construction,Large Family,...,NaN,"277,540.00",122.51,1.77,1.00,883.00,"153,792.00","2,719.00","129,834.00","15,333.00"
4,CA-1990-018,Yucaipa Terrace,San Bernardino,Yucaipa,87.04,"Housing Partners, Inc.",1991,9%,New Construction,Senior,...,NaN,"289,367.00",120.38,0.64,0.00,602.00,"77,423.00","1,584.00","294,246.00","8,860.00"


In [27]:
# Diagnose unmatched ACS counties
unmatched_acs = df[df['county_median_income'].isna()]['county'].value_counts()
if len(unmatched_acs) > 0:
    print('Counties missing ACS data:')
    print(unmatched_acs)
else:
    print('All counties matched to ACS.')

All counties matched to ACS.


### 5. Final Dataset — Validation & Summary

In [28]:
print('=== Final Dataset Shape ===')
print(f'{df.shape[0]:,} rows X {df.shape[1]:,} columns')

print('\n=== Column List ===')
for col in df.columns:
    n_missing = df[col].isna().sum()
    pct = n_missing/len(df) * 100
    print(f' {col: <35} missing: {n_missing: > 5} ({pct:.2f}%)')

=== Final Dataset Shape ===
5,496 rows X 35 columns

=== Column List ===
 app_number                          missing:     0 (0.00%)
 project_name                        missing:     0 (0.00%)
 county                              missing:     0 (0.00%)
 city                                missing:     0 (0.00%)
 census_tract                        missing:     8 (0.15%)
 developer                           missing:   422 (7.68%)
 pis_year                            missing:    47 (0.86%)
 credit_type                         missing:     0 (0.00%)
 construction_type                   missing:     0 (0.00%)
 housing_type                        missing:     0 (0.00%)
 region                              missing:     0 (0.00%)
 total_units                         missing:     0 (0.00%)
 li_units                            missing:     0 (0.00%)
 li_units_pct                        missing:     0 (0.00%)
 studio_pct                          missing:     0 (0.00%)
 one_br_pct                

In [29]:
# Enrichment feature check: confirm new columns have reasonable values
enrichment_cols = [
    'ppi_at_allocation', 'ppi_yoy_change', 'ppi_2yr_trend',
    'fmr_2br',
    'county_median_income', 'county_median_rent',
    'county_poverty_count', 'county_rental_vacancy']
df[enrichment_cols].describe()

,ppi_at_allocation,ppi_yoy_change,ppi_2yr_trend,fmr_2br,county_median_income,county_median_rent,county_poverty_count,county_rental_vacancy
count,"5,151.00","5,149.00","5,151.00","5,151.00","5,496.00","5,495.00","5,496.00","5,496.00"
mean,220.44,3.33,0.87,"1,465.82","95,695.55","1,868.65","433,372.80","24,905.85"
std,67.82,5.66,0.33,698.14,"25,057.61",424.37,"519,602.22","29,045.45"
min,119.62,-3.70,0.00,423.00,"53,847.00",796.00,208.00,0.00
25%,169.56,0.16,1.00,921.00,"83,411.00","1,599.00","74,315.00","3,547.00"
50%,209.28,2.15,1.00,"1,332.00","84,505.00","1,805.00","193,675.00","13,169.00"
75%,239.18,4.08,1.00,"1,816.00","109,361.00","2,229.00","338,752.00","21,644.00"
max,349.95,26.85,1.00,"4,223.00","153,792.00","2,805.00","1,343,978.00","75,430.00"


In [30]:
# Confirm target variable is intact
award= df['annual_federal_award']

print(f'Target: annual_federal_award')
print(f'Non null:  {award.notna().sum():,}')
print(f'Award > 0: {(award>0).sum():,}')
print(f'Median:   ${award.median():,.0f}')
print(f'Min:      ${award.min():,.0f}')
print(f'Max:      ${award.max():,.0f}')

Target: annual_federal_award
Non null:  5,496
Award > 0: 5,496
Median:   $855,592
Min:      $11,119
Max:      $16,529,220


### 6. Save Final Merged Dataset

In [31]:
out_path = 'processed/lihtc_ca_clean.parquet'

# Ensure all string columns are properly typed and covert to string
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str)

# Use pyarrow engine instead (more robust with string encoding)
df.to_parquet(out_path, engine='pyarrow', index=False)

print(f'Saved: {out_path}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Verify round-trip
verify = pd.read_parquet(out_path, engine='pyarrow')
assert verify.shape == df.shape, 'Round-trip shape mismatch!'
print('Round-trip verification: OK')

Saved: processed/lihtc_ca_clean.parquet
Shape: 5,496 rows × 35 columns
Round-trip verification: OK


---
### 7. Feature Summary for Modeling

Quick reference of all model-ready features now in `lihtc_ca_clean.parquet`.

In [32]:
df.columns

Index(['app_number', 'project_name', 'county', 'city', 'census_tract',
       'developer', 'pis_year', 'credit_type', 'construction_type',
       'housing_type', 'region', 'total_units', 'li_units', 'li_units_pct',
       'studio_pct', 'one_br_pct', 'two_br_pct', 'three_plus_br_pct',
       'deep_ami_pct', 'low_ami_pct', 'mid_ami_pct', 'high_ami_pct',
       'credit_per_unit', 'implied_eligible_basis', 'implied_basis_per_unit',
       'total_state_award', 'annual_federal_award', 'ppi_at_allocation',
       'ppi_yoy_change', 'ppi_2yr_trend', 'fmr_2br', 'county_median_income',
       'county_median_rent', 'county_poverty_count', 'county_rental_vacancy'],
      dtype='str')

In [33]:
feature_map = {
    # CTCAC base features
    'credit_type':            'Categorical  | 9% vs 4% credit program',
    'construction_type':      'Categorical  | New Construction vs Rehab',
    'housing_type':           'Categorical  | Large Family, Senior, SRO, etc.',
    'region':                 'Categorical  | 5 CA regions (Bay Area, LA Metro, etc.)',
    'county':                 'Categorical  | 58 CA counties',
    'total_units':            'Numeric      | Project size',
    'li_units_pct':           'Numeric      | Low-income unit fraction',
    'studio_pct':             'Numeric      | SRO/Studio fraction',
    'one_br_pct':             'Numeric      | 1BR fraction',
    'two_br_pct':             'Numeric      | 2BR fraction',
    'three_plus_br_pct':      'Numeric      | 3BR+ fraction',
    'deep_ami_pct':           'Numeric      | Units at ≤30% AMI fraction',
    'mid_ami_pct':            'Numeric      | Units at 50–60% AMI fraction',
    'pis_year':               'Numeric      | Year placed in service (time trend)',
    # PPI features
    'ppi_at_allocation':      'Numeric      | Annual avg PPI in pis_year',
    'ppi_yoy_change':         'Numeric      | PPI % change YoY',
    'ppi_2yr_trend':          'Binary       | PPI rising vs flat/falling (2yr)',
    # FMR features
    'fmr_2br':                'Numeric      | HUD 2BR FMR for county × year',
    # ACS features
    'county_median_income':   'Numeric      | ACS 2022 median HH income',
    'county_median_rent':     'Numeric      | ACS 2022 median gross rent',
    'county_poverty_count':   'Numeric      | ACS 2022 below-poverty population',
    'county_rental_vacancy':  'Numeric      | ACS 2022 rental vacancy count',
    # Target variables
    'annual_federal_award':   'TARGET       | Annual federal tax credit award',
    'total_state_award':      'TARGET (opt) | Annual state tax credit award',
    # Derived display outputs
    'implied_eligible_basis': 'DISPLAY      | Derived cost proxy',
    'implied_basis_per_unit': 'DISPLAY      | Per-unit cost proxy',
    'credit_per_unit':        'DISPLAY      | Annual federal credit per unit',
}

print(f'Total features: {len(feature_map)}')
print()
for feat, desc in feature_map.items():
    in_df = '✓' if feat in df.columns else '✗ MISSING'
    print(f'  [{in_df}] {feat:<30}  {desc}')

Total features: 27

  [✓] credit_type                     Categorical  | 9% vs 4% credit program
  [✓] construction_type               Categorical  | New Construction vs Rehab
  [✓] housing_type                    Categorical  | Large Family, Senior, SRO, etc.
  [✓] region                          Categorical  | 5 CA regions (Bay Area, LA Metro, etc.)
  [✓] county                          Categorical  | 58 CA counties
  [✓] total_units                     Numeric      | Project size
  [✓] li_units_pct                    Numeric      | Low-income unit fraction
  [✓] studio_pct                      Numeric      | SRO/Studio fraction
  [✓] one_br_pct                      Numeric      | 1BR fraction
  [✓] two_br_pct                      Numeric      | 2BR fraction
  [✓] three_plus_br_pct               Numeric      | 3BR+ fraction
  [✓] deep_ami_pct                    Numeric      | Units at ≤30% AMI fraction
  [✓] mid_ami_pct                     Numeric      | Units at 50–60% AMI fraction
